# Suivi et Détection des Sites Miniers

Ce notebook permet d'identifier les zones d'exploitation minière en détectant les perturbations du sol et les signatures anthropiques via Prithvi.

In [ ]:
!pip install geemap earthengine-api scikit-learn rasterio terratorch torch matplotlib -q
import ee, geemap, torch, rasterio
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import MiniBatchKMeans
from terratorch import BACKBONE_REGISTRY

ee.Initialize(project='geocongoai-api')

In [ ]:
roi = ee.Geometry.Rectangle([15.1, -4.9, 15.3, -4.7])
image = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(roi).filterDate('2023-01-01', '2023-12-31').median().clip(roi)
geemap.ee_export_image(image.select(['B2', 'B3', 'B4', 'B8', 'B11', 'B12']), 'input.tif', scale=30, region=roi)

In [ ]:
model = BACKBONE_REGISTRY.build("prithvi_eo_v2_300", num_frames=1, in_chans=6, pretrained=True).eval().to('cpu')
with rasterio.open('input.tif') as src: img = src.read().astype(np.float32) / 10000.0
with torch.no_grad():
    out = model(torch.from_numpy(img).unsqueeze(0))
    feats = out[0] if isinstance(out, list) else out
    
feats_np = feats[0, 1:].numpy()
kmeans = MiniBatchKMeans(n_clusters=4).fit(feats_np)
mining_labels = kmeans.labels_.reshape(int(np.sqrt(feats_np.shape[0])), -1)

plt.imshow(mining_labels, cmap='Set1')
plt.title("Détection des Zones Minières")
plt.show()